# Similarity Search

En este notebook se implementa una búsqueda de similitud acústica entre canciones.

Hasta ahora, el proyecto construyó una lista de `hidden gems`: canciones poco populares, recientes, de artistas no masivos y con alto perfil acústico de hit.

El objetivo ahora es pasar de:

> “Esta canción tiene alto perfil acústico”

a:

> “Esta canción poco popular se parece acústicamente a este hit específico”.

Para ello, cada canción se representará como un vector acústico usando las mismas variables del `Hit Score V2`:

- `danceability_norm`
- `energy_norm`
- `loudness_norm`
- `acousticness_norm`
- `instrumentalness_norm`

La similitud se medirá usando distancia euclidiana. Las ponderaciones corresponden a los pesos usados en el `Hit Score V2`, de forma que las variables más relacionadas con popularidad tengan mayor influencia en la distancia.

El notebook tendrá dos etapas:

1. Búsqueda exacta sobre una muestra pequeña, para explicar y validar el concepto.
2. Búsqueda aproximada con LSH, para conectar con escalabilidad y datos masivos.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler
import math

spark = (
    SparkSession.builder
    .appName("HitMakerAI_SimilaritySearch")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/21 23:01:45 WARN Utils: Your hostname, debian, resolves to a loopback address: 127.0.1.1; using 192.168.100.25 instead (on interface wlp3s0)
26/05/21 23:01:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/21 23:01:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
tracks_enriched = spark.read.parquet(
    "../processed/tracks_enriched_artists.parquet"
)

hidden_gems = spark.read.parquet(
    "../processed/hidden_gems_v3_recency.parquet"
)

In [3]:
from pyspark.sql import functions as F

W = {
    "acousticness": 0.25988213907708085,
    "loudness": 0.23016439975443223,
    "energy": 0.21218715648715247,
    "instrumentalness": 0.16620344705647533,
    "danceability": 0.13156285762485917
}

tracks_enriched = (
    tracks_enriched
    .withColumn(
        "hit_score_weighted",
        F.round(
            (
                (1 - F.col("acousticness_norm")) * F.lit(W["acousticness"]) +
                F.col("loudness_norm") * F.lit(W["loudness"]) +
                F.col("energy_norm") * F.lit(W["energy"]) +
                (1 - F.col("instrumentalness_norm")) * F.lit(W["instrumentalness"]) +
                F.col("danceability_norm") * F.lit(W["danceability"])
            ) * 100,
            2
        )
    )
)

In [4]:
tracks_enriched.select(
    "track_id",
    "track_name",
    "popularity",
    "is_hit",
    "release_year",
    "hit_score",
    "hit_score_weighted"
).show(10, truncate=False, vertical=True)

-RECORD 0---------------------------------------------------------------------
 track_id           | 00AeAaSNbe92PRrstQskvH                                  
 track_name         | Přítel Fořt                                             
 popularity         | 3                                                       
 is_hit             | 0                                                       
 release_year       | 1980                                                    
 hit_score          | 65.54                                                   
 hit_score_weighted | 63.28                                                   
-RECORD 1---------------------------------------------------------------------
 track_id           | 00DJt4PjkzeXhKKVDekw2n                                  
 track_name         | Music Man                                               
 popularity         | 9                                                       
 is_hit             | 0                             

In [5]:
similarity_features = [
    "danceability_norm",
    "energy_norm",
    "loudness_norm",
    "acousticness_norm",
    "instrumentalness_norm"
]

tracks_feature_ref = tracks_enriched.select(
    "track_id",
    *similarity_features
)

In [6]:
hidden_gems = (
    hidden_gems
    .drop(*[c for c in similarity_features if c in hidden_gems.columns])
    .join(
        tracks_feature_ref,
        on="track_id",
        how="left"
    )
)

In [7]:
hidden_gems.select(
    "track_id",
    "track_name",
    "popularity",
    "release_year",
    "hit_score_weighted",
    "hidden_gem_score"
).show(10, truncate=False)

+----------------------+---------------------------------+----------+------------+------------------+----------------+
|track_id              |track_name                       |popularity|release_year|hit_score_weighted|hidden_gem_score|
+----------------------+---------------------------------+----------+------------+------------------+----------------+
|1fnkpqws9eYHsspp226IU3|Frumoasa Și Bestia               |25        |2010        |87.6              |79.41           |
|3rthHkmV7fOq2EnjBQa6e3|Tervaskanto                      |37        |2013        |84.9              |79.41           |
|5ybujObJcKYZacDS5V2Qci|Vi Ses I helvede                 |31        |2011        |86.76             |79.41           |
|1spVlPNUbPFdiQ9TBkct05|Chemical Invasion - 2005 Remaster|2         |2016        |81.59             |79.41           |
|0S1vxen26XzaAkTuqkqcHq|Enkeli (feat. Väinöväinö)        |37        |2013        |84.9              |79.41           |
|56OEe3VdyyOnkFBNWkc4FB|Eu Quero Bem À Minha Sog

In [8]:
hidden_gems.select(
    F.count("*").alias("total_hidden_gems"),
    F.sum(F.col("danceability_norm").isNull().cast("int")).alias("missing_norm_features")
).show()

[Stage 6:============================================>              (6 + 2) / 8]

+-----------------+---------------------+
|total_hidden_gems|missing_norm_features|
+-----------------+---------------------+
|            13176|                    0|
+-----------------+---------------------+



In [9]:
hits_modern = (
    tracks_enriched
    .filter(
        (F.col("is_hit") == 1) &
        (F.col("release_year") >= 2010) &
        (F.col("popularity") >= 70)
    )
)

In [10]:
version_pattern = r"(?i)(karaoke|cover|re-recorded|re recorded|live|tribute|workout|version|mix cut)"

hidden_gems_clean = (
    hidden_gems
    .filter(F.col("popularity") > 0)
    .filter(~F.col("track_name").rlike(version_pattern))
)

In [11]:
# Hidden gems limpias
hidden_gems_clean.select(
    F.count("*").alias("num_hidden_gems_clean"),
    F.min("release_year").alias("min_release_year"),
    F.max("release_year").alias("max_release_year"),
    F.min("popularity").alias("min_popularity"),
    F.max("popularity").alias("max_popularity"),
    F.min("hit_score_weighted").alias("min_hit_score_weighted"),
    F.max("hit_score_weighted").alias("max_hit_score_weighted"),
    F.min("hidden_gem_score").alias("min_hidden_gem_score"),
    F.max("hidden_gem_score").alias("max_hidden_gem_score")
).show(vertical=True)

-RECORD 0-----------------------
 num_hidden_gems_clean  | 11613 
 min_release_year       | 2010  
 max_release_year       | 2021  
 min_popularity         | 1     
 max_popularity         | 39    
 min_hit_score_weighted | 80.0  
 max_hit_score_weighted | 93.82 
 min_hidden_gem_score   | 73.33 
 max_hidden_gem_score   | 92.77 



In [12]:
hidden_gems_clean.select(
    "track_id",
    "track_name",
    "artists",
    "release_year",
    "popularity",
    "hit_score_weighted",
    "hidden_gem_score"
).orderBy(F.desc("hidden_gem_score")).show(20, truncate=False, vertical=True)

-RECORD 0---------------------------------------------------------------------------------
 track_id           | 6rGaZ6NAMkXrTBN0H72w1D                                              
 track_name         | Predestination                                                      
 artists            | ['Ryan']                                                            
 release_year       | 2020                                                                
 popularity         | 39                                                                  
 hit_score_weighted | 92.64                                                               
 hidden_gem_score   | 92.77                                                               
-RECORD 1---------------------------------------------------------------------------------
 track_id           | 1qODPPzPKUFPMzW43L5udq                                              
 track_name         | Linnud                                                              

filtrar hits con columnas completas

In [13]:
similarity_features = [
    "danceability_norm",
    "energy_norm",
    "loudness_norm",
    "acousticness_norm",
    "instrumentalness_norm"
]

hits_modern_clean = hits_modern.dropna(subset=similarity_features)
hidden_gems_clean = hidden_gems_clean.dropna(subset=similarity_features)

In [14]:
hits_modern_clean.select(
    F.count("*").alias("num_hits_modern_clean")
).show()

hidden_gems_clean.select(
    F.count("*").alias("num_hidden_gems_clean")
).show()

+---------------------+
|num_hits_modern_clean|
+---------------------+
|                 5564|
+---------------------+

+---------------------+
|num_hidden_gems_clean|
+---------------------+
|                11613|
+---------------------+



Eso significa que una búsqueda exacta completa sería:

5,564 × 11,613 ≈ 64.6 millones de comparaciones

Eso ya justifica por qué primero haremos una muestra exacta y luego LSH.

In [15]:
import math
from pyspark.sql import functions as F

feature_weights = {
    "danceability_norm": 0.13156285762485917,
    "energy_norm": 0.21218715648715247,
    "loudness_norm": 0.23016439975443223,
    "acousticness_norm": 0.25988213907708085,
    "instrumentalness_norm": 0.16620344705647533
}

def add_weighted_features(df):
    result = df
    for feature, weight in feature_weights.items():
        result = result.withColumn(
            f"{feature}_w",
            F.col(feature) * F.lit(math.sqrt(weight))
        )
    return result

hits_features = add_weighted_features(hits_modern_clean)
gems_features = add_weighted_features(hidden_gems_clean)

weighted_feature_cols = [f"{feature}_w" for feature in similarity_features]

In [16]:
hits_features.select(
    "track_name",
    "popularity",
    *weighted_feature_cols
).show(5, truncate=False)

gems_features.select(
    "track_name",
    "popularity",
    *weighted_feature_cols
).show(5, truncate=False, vertical=True)

+-------------------------------------+----------+-------------------+-------------------+-------------------+--------------------+-----------------------+
|track_name                           |popularity|danceability_norm_w|energy_norm_w      |loudness_norm_w    |acousticness_norm_w |instrumentalness_norm_w|
+-------------------------------------+----------+-------------------+-------------------+-------------------+--------------------+-----------------------+
|炎                                   |78        |0.1639144998910904 |0.31553687344379283|0.40084961791733614|0.053742538596496414|0.0                    |
|At My Worst                          |86        |0.29387030914877416|0.19116467515207886|0.38994755084001415|0.3976947856140735  |0.0                    |
|Monsters (feat. blackbear)           |79        |0.12214298977254918|0.3892389168759196 |0.40963007281561664|0.025233401455307364|0.0                    |
|The Woo (feat. 50 Cent & Roddy Ricch)|83        |0.1689425520349

## Construir vectores con VectorAssembler

In [17]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=weighted_feature_cols,
    outputCol="features"
)

hits_vector = assembler.transform(hits_features)
gems_vector = assembler.transform(gems_features)

In [18]:
hits_vector.select(
    "track_id",
    "track_name",
    "artists",
    "features"
).show(5, truncate=False)

gems_vector.select(
    "track_id",
    "track_name",
    "artists",
    "features"
).show(5, truncate=False)

+----------------------+-------------------------------------+---------------------------------------+------------------------------------------------------------------------------------------------------+
|track_id              |track_name                           |artists                                |features                                                                                              |
+----------------------+-------------------------------------+---------------------------------------+------------------------------------------------------------------------------------------------------+
|0cSkn2l67csUljEy0EEBPn|炎                                   |['LiSA']                               |[0.1639144998910904,0.31553687344379283,0.40084961791733614,0.053742538596496414,0.0]                 |
|0ri0Han4IRJhzvERHOZTMr|At My Worst                          |['Pink Sweat$']                        |[0.29387030914877416,0.19116467515207886,0.38994755084001415,0.397694785614

### Representación vectorial ponderada

Para calcular similitud acústica, cada canción se representa como un vector numérico usando las cinco variables del `Hit Score V2`:

- `danceability_norm`
- `energy_norm`
- `loudness_norm`
- `acousticness_norm`
- `instrumentalness_norm`

Estas variables ya están normalizadas, por lo que pueden compararse dentro de una misma escala.

Además, se aplican los pesos del `Hit Score V2` para que las variables con mayor relación observada con popularidad tengan mayor influencia en la distancia.

Para implementar distancia euclidiana ponderada, cada variable se multiplica por la raíz cuadrada de su peso. De esta forma, al calcular distancia euclidiana estándar sobre los vectores transformados, se obtiene una distancia ponderada.

## Búsqueda exacta en muestra pequeña

Antes de usar un método aproximado como LSH, se implementa una búsqueda exacta sobre una muestra pequeña.

El objetivo es validar el concepto de similitud acústica de forma transparente: se comparan hidden gems contra hits modernos y se calcula la distancia euclidiana ponderada entre sus vectores acústicos.

En esta primera prueba se usan:

- 500 hits modernos con mayor popularidad;
- 500 hidden gems con mayor `hidden_gem_score`.

Esto produce 250,000 comparaciones, una cantidad manejable para revisar resultados y validar si los pares encontrados tienen sentido.

In [19]:
hits_sample = (
    hits_vector
    .orderBy(F.desc("popularity"))
    .limit(500)
)

gems_sample = (
    gems_vector
    .orderBy(F.desc("hidden_gem_score"))
    .limit(500)
)

In [20]:
hits_sample.select(F.count("*").alias("num_hits_sample")).show()
gems_sample.select(F.count("*").alias("num_gems_sample")).show()

+---------------+
|num_hits_sample|
+---------------+
|            500|
+---------------+

+---------------+
|num_gems_sample|
+---------------+
|            500|
+---------------+



In [21]:
#Renombramos columnas para el cross join 
hit_cols = [
    F.col("track_id").alias("hit_track_id"),
    F.col("track_name").alias("hit_name"),
    F.col("artists").alias("hit_artists"),
    F.col("release_year").alias("hit_release_year"),
    F.col("popularity").alias("hit_popularity"),
    F.col("hit_score_weighted").alias("hit_score_weighted")
] + [
    F.col(c).alias(f"hit_{c}") for c in weighted_feature_cols
]

gem_cols = [
    F.col("track_id").alias("gem_track_id"),
    F.col("track_name").alias("gem_name"),
    F.col("artists").alias("gem_artists"),
    F.col("release_year").alias("gem_release_year"),
    F.col("popularity").alias("gem_popularity"),
    F.col("hit_score_weighted").alias("gem_hit_score_weighted"),
    F.col("hidden_gem_score").alias("gem_hidden_gem_score")
] + [
    F.col(c).alias(f"gem_{c}") for c in weighted_feature_cols
]

hits_sample_renamed = hits_sample.select(*hit_cols)
gems_sample_renamed = gems_sample.select(*gem_cols)

In [22]:
#Calculamos distancia Euclideana 
distance_expr = None

for c in weighted_feature_cols:
    diff_sq = F.pow(F.col(f"gem_{c}") - F.col(f"hit_{c}"), 2)
    distance_expr = diff_sq if distance_expr is None else distance_expr + diff_sq

similarity_exact_sample = (
    gems_sample_renamed
    .crossJoin(hits_sample_renamed)
    .filter(F.col("gem_artists") != F.col("hit_artists"))
    .withColumn("distance", F.sqrt(distance_expr))
)

In [23]:
similarity_exact_sample.select(
    F.count("*").alias("num_comparisons")
).show()

+---------------+
|num_comparisons|
+---------------+
|         250000|
+---------------+



Elegir el hit más cercanano para cada hidden gem

In [24]:
nearest_hit_window = Window.partitionBy("gem_track_id").orderBy(F.asc("distance"))

hidden_gem_nearest_hit_exact_sample = (
    similarity_exact_sample
    .withColumn("similarity_rank", F.row_number().over(nearest_hit_window))
    .filter(F.col("similarity_rank") == 1)
    .select(
        "gem_track_id",
        "gem_name",
        "gem_artists",
        "gem_release_year",
        "gem_popularity",
        "gem_hit_score_weighted",
        "gem_hidden_gem_score",
        "hit_track_id",
        "hit_name",
        "hit_artists",
        "hit_release_year",
        "hit_popularity",
        "hit_score_weighted",
        "distance",
        "similarity_rank"
    )
    .orderBy(F.asc("distance"))
)

hidden_gem_nearest_hit_exact_sample.show(30, truncate=False, vertical=True)

-RECORD 0------------------------------------------------------------------------------------------------------------
 gem_track_id           | 1B8YgUyohyscDUw7Fhby6E                                                                     
 gem_name               | Beyond The Comfort Zone                                                                    
 gem_artists            | ['Elysian']                                                                                
 gem_release_year       | 2020                                                                                       
 gem_popularity         | 27                                                                                         
 gem_hit_score_weighted | 86.75                                                                                      
 gem_hidden_gem_score   | 88.06                                                                                      
 hit_track_id           | 0AUyNF6iFxMNQsNx2nhtrw        

In [25]:
hidden_gem_nearest_hit_exact_sample.select(
    F.count("*").alias("num_pairs"),
    F.round(F.min("distance"), 4).alias("min_distance"),
    F.round(F.expr("percentile_approx(distance, 0.25)"), 4).alias("p25_distance"),
    F.round(F.expr("percentile_approx(distance, 0.50)"), 4).alias("median_distance"),
    F.round(F.expr("percentile_approx(distance, 0.75)"), 4).alias("p75_distance"),
    F.round(F.max("distance"), 4).alias("max_distance")
).show()

+---------+------------+------------+---------------+------------+------------+
|num_pairs|min_distance|p25_distance|median_distance|p75_distance|max_distance|
+---------+------------+------------+---------------+------------+------------+
|      500|      0.0037|      0.0149|         0.0215|      0.0311|      0.1368|
+---------+------------+------------+---------------+------------+------------+



### Distribución de distancias exactas

Después de seleccionar el hit más cercano para cada hidden gem de la muestra, se analizó la distribución de distancias.

La distancia mínima fue 0.0037, la mediana fue 0.0215 y el percentil 75 fue 0.0311. Estos valores indican que muchas hidden gems tienen al menos un hit moderno muy cercano en el espacio acústico definido por las cinco variables seleccionadas.

Es importante notar que estas distancias corresponden al vecino más cercano de cada hidden gem dentro de una muestra de 500 hits. Por lo tanto, no representan todas las comparaciones posibles, sino únicamente los mejores emparejamientos encontrados para cada canción.

La distancia máxima fue 0.1368, lo que sugiere que algunas hidden gems tienen un match menos cercano, incluso después de buscar su vecino más parecido dentro de la muestra.

In [26]:
#la correlación entre distancia y gem_hit_score_weighted:
hidden_gem_nearest_hit_exact_sample.select(
    F.round(
        F.corr("distance", "gem_hit_score_weighted"),
        4
    ).alias("corr_distance_gem_hit_score_weighted")
).show()

+------------------------------------+
|corr_distance_gem_hit_score_weighted|
+------------------------------------+
|                              0.1273|
+------------------------------------+



### Relación entre distancia y Hit Score

Se calculó la correlación entre la distancia al hit más cercano y el `hit_score_weighted` de la hidden gem.

El resultado fue una correlación baja y positiva de 0.1273. Esto indica que, dentro de esta muestra, no hay una relación lineal fuerte entre tener mayor `Hit Score` y estar más cerca de un hit específico.

Esto es metodológicamente razonable porque ambas métricas responden preguntas diferentes. El `Hit Score` mide alineación con el perfil acústico general de canciones populares, mientras que la distancia mide similitud con una canción hit específica.

Por lo tanto, la búsqueda de similitud no reemplaza al `Hit Score`, sino que lo complementa.

## Segunda parte: búsqueda aproximada con LSH

Después de validar la búsqueda exacta sobre una muestra pequeña, se implementa una versión aproximada usando `BucketedRandomProjectionLSH`.

LSH permite reducir el costo de búsqueda de vecinos cercanos en espacios vectoriales. En este proyecto se usa sobre vectores acústicos ponderados, construidos con las mismas variables del `Hit Score V2`.

Debido a que el notebook se ejecuta en una computadora personal, se utiliza una muestra estratificada. Esta muestra no toma únicamente las canciones mejor rankeadas, sino que conserva diversidad por año de lanzamiento, popularidad y score.

El objetivo es mostrar el método de forma computacionalmente viable y metodológicamente defendible.

In [27]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import BucketedRandomProjectionLSH

In [28]:
def add_year_bucket(df):
    return (
        df
        .withColumn(
            "year_bucket",
            F.when(F.col("release_year") <= 2013, "2010-2013")
             .when(F.col("release_year") <= 2017, "2014-2017")
             .otherwise("2018-2021")
        )
    )

In [29]:
hits_sampling_frame = (
    add_year_bucket(hits_vector)
    .withColumn(
        "hit_popularity_bin",
        F.when(F.col("popularity") < 80, "70-79")
         .when(F.col("popularity") < 90, "80-89")
         .otherwise("90-100")
    )
    .withColumn(
        "sample_stratum",
        F.concat_ws("_", "year_bucket", "hit_popularity_bin")
    )
)

In [30]:
gems_sampling_frame = (
    add_year_bucket(gems_vector)
    .withColumn(
        "gem_score_bin",
        F.when(F.col("hidden_gem_score") < 85, "80-84")
         .when(F.col("hidden_gem_score") < 90, "85-89")
         .otherwise("90-100")
    )
    .withColumn(
        "gem_popularity_bin",
        F.when(F.col("popularity") < 10, "01-09")
         .when(F.col("popularity") < 20, "10-19")
         .when(F.col("popularity") < 30, "20-29")
         .otherwise("30-39")
    )
    .withColumn(
        "sample_stratum",
        F.concat_ws("_", "year_bucket", "gem_score_bin", "gem_popularity_bin")
    )
)

In [31]:
#Función de muestreo estratificado
def proportional_stratified_sample(df, stratum_col, target_n, seed=42):
    total_n = df.count()

    strata_counts = (
        df
        .groupBy(stratum_col)
        .agg(F.count("*").alias("stratum_count"))
        .withColumn(
            "target_stratum_n",
            F.greatest(
                F.lit(1),
                F.round(
                    F.col("stratum_count") / F.lit(total_n) * F.lit(target_n)
                ).cast("int")
            )
        )
    )

    w = Window.partitionBy(stratum_col).orderBy(F.rand(seed))

    sampled = (
        df
        .join(
            strata_counts.select(stratum_col, "target_stratum_n"),
            on=stratum_col,
            how="left"
        )
        .withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") <= F.col("target_stratum_n"))
        .drop("rn", "target_stratum_n")
    )

    return sampled

In [32]:
#crear muestras pequeñas pero representativas
hits_lsh_sample = proportional_stratified_sample(
    hits_sampling_frame,
    "sample_stratum",
    target_n=800,
    seed=42
)

gems_lsh_sample = proportional_stratified_sample(
    gems_sampling_frame,
    "sample_stratum",
    target_n=1200,
    seed=42
)

In [33]:
#reducimos columnas para que spark cargue menos
hits_lsh_sample = (
    hits_lsh_sample
    .select(
        "track_id",
        "track_name",
        "artists",
        "release_year",
        "popularity",
        "hit_score_weighted",
        "features",
        "sample_stratum"
    )
)

gems_lsh_sample = (
    gems_lsh_sample
    .select(
        "track_id",
        "track_name",
        "artists",
        "release_year",
        "popularity",
        "hit_score_weighted",
        "hidden_gem_score",
        "features",
        "sample_stratum"
    )
)

In [34]:
hits_lsh_sample.select(F.count("*").alias("num_hits_lsh_sample")).show()
gems_lsh_sample.select(F.count("*").alias("num_gems_lsh_sample")).show()

+-------------------+
|num_hits_lsh_sample|
+-------------------+
|                801|
+-------------------+

+-------------------+
|num_gems_lsh_sample|
+-------------------+
|               1204|
+-------------------+



### Validar representatividad

In [35]:
hits_profile_full = hits_sampling_frame.select(
    F.lit("full_hits").alias("dataset"),
    F.count("*").alias("n"),
    F.round(F.avg("release_year"), 2).alias("avg_release_year"),
    F.round(F.avg("popularity"), 2).alias("avg_popularity"),
    F.round(F.avg("hit_score_weighted"), 2).alias("avg_hit_score_weighted")
)

hits_profile_sample = hits_lsh_sample.select(
    F.lit("sample_hits").alias("dataset"),
    F.count("*").alias("n"),
    F.round(F.avg("release_year"), 2).alias("avg_release_year"),
    F.round(F.avg("popularity"), 2).alias("avg_popularity"),
    F.round(F.avg("hit_score_weighted"), 2).alias("avg_hit_score_weighted")
)

hits_profile_full.unionByName(hits_profile_sample).show(truncate=False)

+-----------+----+----------------+--------------+----------------------+
|dataset    |n   |avg_release_year|avg_popularity|avg_hit_score_weighted|
+-----------+----+----------------+--------------+----------------------+
|full_hits  |5564|2017.97         |74.84         |76.18                 |
|sample_hits|801 |2017.98         |74.75         |75.57                 |
+-----------+----+----------------+--------------+----------------------+



In [36]:
gems_profile_full = gems_sampling_frame.select(
    F.lit("full_gems").alias("dataset"),
    F.count("*").alias("n"),
    F.round(F.avg("release_year"), 2).alias("avg_release_year"),
    F.round(F.avg("popularity"), 2).alias("avg_popularity"),
    F.round(F.avg("hit_score_weighted"), 2).alias("avg_hit_score_weighted"),
    F.round(F.avg("hidden_gem_score"), 2).alias("avg_hidden_gem_score")
)

gems_profile_sample = gems_lsh_sample.select(
    F.lit("sample_gems").alias("dataset"),
    F.count("*").alias("n"),
    F.round(F.avg("release_year"), 2).alias("avg_release_year"),
    F.round(F.avg("popularity"), 2).alias("avg_popularity"),
    F.round(F.avg("hit_score_weighted"), 2).alias("avg_hit_score_weighted"),
    F.round(F.avg("hidden_gem_score"), 2).alias("avg_hidden_gem_score")
)

gems_profile_full.unionByName(gems_profile_sample).show(truncate=False)

+-----------+-----+----------------+--------------+----------------------+--------------------+
|dataset    |n    |avg_release_year|avg_popularity|avg_hit_score_weighted|avg_hidden_gem_score|
+-----------+-----+----------------+--------------+----------------------+--------------------+
|full_gems  |11613|2014.29         |26.71         |84.98                 |80.83               |
|sample_gems|1204 |2014.3          |26.81         |84.86                 |80.74               |
+-----------+-----+----------------+--------------+----------------------+--------------------+



Para evitar que la demostración de LSH dependiera únicamente de los hits más populares, se construyó una muestra estratificada por año de lanzamiento y rango de popularidad.

Al comparar la muestra con el conjunto completo de hits modernos, se observa que los promedios son muy similares. El año promedio de lanzamiento, la popularidad promedio y el `hit_score_weighted` promedio prácticamente se mantienen.

Esto sugiere que la muestra de hits conserva razonablemente la estructura del conjunto completo, por lo que es adecuada para una demostración reducida de LSH.

### Validación de representatividad de la muestra LSH

La muestra usada para LSH fue construida mediante muestreo estratificado, no mediante selección de los primeros registros o de los casos con mayor score.

Para los hits, se estratificó por año de lanzamiento y rango de popularidad. Para las hidden gems, se estratificó por año de lanzamiento, rango de popularidad y rango de `hidden_gem_score`.

Después de comparar los promedios de la muestra contra los conjuntos completos, se observó que las variables principales mantienen valores similares. Esto sugiere que la muestra conserva razonablemente la estructura de los datos originales y puede usarse como una demostración representativa de LSH en una computadora personal.

### LSH

In [37]:
brp_lsh = BucketedRandomProjectionLSH(
    inputCol="features",
    outputCol="hashes",
    bucketLength=0.25,
    numHashTables=1
)

brp_lsh_model = brp_lsh.fit(hits_lsh_sample)

hits_lsh_hashed = brp_lsh_model.transform(hits_lsh_sample)
gems_lsh_hashed = brp_lsh_model.transform(gems_lsh_sample)

In [38]:
LSH_DISTANCE_THRESHOLD = 0.15

lsh_candidates_raw = brp_lsh_model.approxSimilarityJoin(
    gems_lsh_hashed,
    hits_lsh_hashed,
    LSH_DISTANCE_THRESHOLD,
    distCol="distance"
)

In [39]:
lsh_candidates = (
    lsh_candidates_raw
    .select(
        F.col("datasetA.track_id").alias("gem_track_id"),
        F.col("datasetA.track_name").alias("gem_name"),
        F.col("datasetA.artists").alias("gem_artists"),
        F.col("datasetA.release_year").alias("gem_release_year"),
        F.col("datasetA.popularity").alias("gem_popularity"),
        F.col("datasetA.hit_score_weighted").alias("gem_hit_score_weighted"),
        F.col("datasetA.hidden_gem_score").alias("gem_hidden_gem_score"),
        F.col("datasetB.track_id").alias("hit_track_id"),
        F.col("datasetB.track_name").alias("hit_name"),
        F.col("datasetB.artists").alias("hit_artists"),
        F.col("datasetB.release_year").alias("hit_release_year"),
        F.col("datasetB.popularity").alias("hit_popularity"),
        F.col("datasetB.hit_score_weighted").alias("hit_score_weighted"),
        F.col("distance")
    )
    .filter(F.col("gem_artists") != F.col("hit_artists"))
)

In [40]:
lsh_candidates.show(20, truncate=False, vertical=True)

[Stage 124:>                                                        (0 + 1) / 1]

-RECORD 0--------------------------------------------------------------------------
 gem_track_id           | 5cdgjSzxt0cHmIYi2F6o9x                                   
 gem_name               | Timing - 2012 Remaster                                   
 gem_artists            | ['Gentle Giant']                                         
 gem_release_year       | 2012                                                     
 gem_popularity         | 2                                                        
 gem_hit_score_weighted | 82.13                                                    
 gem_hidden_gem_score   | 76.42                                                    
 hit_track_id           | 1Ej96GIBCTvgH7tNX1r3qr                                   
 hit_name               | Otro Trago                                               
 hit_artists            | ['Sech', 'Darell']                                       
 hit_release_year       | 2019                                              

### Elegir el vecino más cercano por Hiden gem

In [41]:
nearest_lsh_window = Window.partitionBy("gem_track_id").orderBy(F.asc("distance"))

hidden_gem_nearest_hit_lsh = (
    lsh_candidates
    .withColumn("similarity_rank", F.row_number().over(nearest_lsh_window))
    .filter(F.col("similarity_rank") == 1)
    .select(
        "gem_track_id",
        "gem_name",
        "gem_artists",
        "gem_release_year",
        "gem_popularity",
        "gem_hit_score_weighted",
        "gem_hidden_gem_score",
        "hit_track_id",
        "hit_name",
        "hit_artists",
        "hit_release_year",
        "hit_popularity",
        "hit_score_weighted",
        "distance",
        "similarity_rank"
    )
)

In [42]:
hidden_gem_nearest_hit_lsh.show(30, truncate=False, vertical=True)

[Stage 149:===================================>                     (5 + 3) / 8]

-RECORD 0---------------------------------------------------------------------------------------------------------
 gem_track_id           | 01CUj1noOVz339DHsCTmr6                                                                  
 gem_name               | La Kenworth Plateada                                                                    
 gem_artists            | ['Uriel Henao']                                                                         
 gem_release_year       | 2015                                                                                    
 gem_popularity         | 36                                                                                      
 gem_hit_score_weighted | 80.41                                                                                   
 gem_hidden_gem_score   | 77.52                                                                                   
 hit_track_id           | 5e50NiIlOc2YJIftHzoehd                                

In [43]:
#Resumen ligero
hidden_gem_nearest_hit_lsh.select(
    F.count("*").alias("num_gems_with_lsh_match"),
    F.round(F.min("distance"), 4).alias("min_distance"),
    F.round(F.max("distance"), 4).alias("max_distance")
).show()

[Stage 171:>                                                        (0 + 1) / 1]

+-----------------------+------------+------------+
|num_gems_with_lsh_match|min_distance|max_distance|
+-----------------------+------------+------------+
|                   1204|      0.0038|       0.118|
+-----------------------+------------+------------+



### Escala y representatividad de la muestra LSH

Para que la demostración con LSH fuera más coherente con su objetivo de escalabilidad, se utilizó una muestra mayor que la búsqueda exacta inicial.

La búsqueda exacta se realizó sobre 500 hits y 500 hidden gems, lo que equivale a 250,000 comparaciones potenciales. En cambio, la muestra usada para LSH contiene 801 hits y 1,204 hidden gems, lo que representa 964,404 comparaciones potenciales.

Además, la muestra LSH fue construida mediante muestreo estratificado, no mediante selección de los primeros registros del ranking. Para los hits, se estratificó por año de lanzamiento y rango de popularidad. Para las hidden gems, se estratificó por año de lanzamiento, popularidad y `hidden_gem_score`.

Al comparar los promedios de las muestras contra los conjuntos completos, se observa que los valores son muy similares. Esto sugiere que la muestra conserva razonablemente la estructura del conjunto original.

La búsqueda con LSH encontró un hit cercano para las 1,204 hidden gems de la muestra. Las distancias obtenidas se ubicaron entre 0.0038 y 0.118, lo que muestra que el método logró recuperar vecinos cercanos en el espacio acústico ponderado.

## Guardar datos

In [44]:
hidden_gem_nearest_hit_lsh.write.mode("overwrite").parquet(
    "../processed/hidden_gem_nearest_hit_lsh_sample.parquet"
)

hits_lsh_sample.write.mode("overwrite").parquet(
    "../processed/similarity_hits_lsh_sample.parquet"
)

gems_lsh_sample.write.mode("overwrite").parquet(
    "../processed/similarity_gems_lsh_sample.parquet"
)

## Conclusión del Similarity Search

En este notebook se implementó una búsqueda de similitud acústica entre hidden gems y hits modernos.

Cada canción fue representada como un vector acústico ponderado usando las cinco variables del `Hit Score V2`: `danceability`, `energy`, `loudness`, `acousticness` e `instrumentalness`.

Primero se realizó una búsqueda exacta sobre una muestra pequeña para validar el concepto de similitud. Después se implementó una búsqueda aproximada con `BucketedRandomProjectionLSH` sobre una muestra estratificada representativa.

La búsqueda exacta permitió observar pares concretos de hidden gems e hits con distancias acústicas bajas. La búsqueda con LSH mostró que es posible recuperar candidatos cercanos sin comparar exhaustivamente todos los pares.

El resultado principal del notebook es una tabla donde cada hidden gem se asocia con un hit moderno acústicamente cercano. Esto complementa el `Hit Score`: mientras el score mide alineación con el perfil general de canciones populares, la búsqueda de similitud identifica a qué hit específico se parece cada hidden gem.

La similitud calculada debe interpretarse como similitud acústica aproximada basada en variables de Spotify, no como equivalencia musical completa. El método no considera letra, idioma, género explícito, contexto cultural ni percepción subjetiva.